In [ ]:
import numpy as np
import pandas as pd
import pyBigWig as pbw
g4 = pd.read_csv("/home/maxime/Nucleoseq/data/g4_intra_mm10.bed", sep="\t", header=None)
from sys import path
path.insert(1, "/home/maxime/Nucleoseq/src")
import utils as mf
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
mpl_style = "genome_research.mplstyle"
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.style.use(mpl_style)

mpl_style = REPO_ROOT / "notebooks" / "genome_research.mplstyle"
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.style.use(mpl_style)

plt.rcParams.update({
    # Font — picks the first one it finds installed
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],

    # Force every text element to the same size
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "figure.titlesize": 10,

    # Keep text as real, editable text in the SVG — not outlines
    "svg.fonttype": "none",
})


ncbi_to_ucsc_mapping = {
    "gi|371561115|gb|CM000994.2|": "chr1",
    "gi|371561114|gb|CM000995.2|": "chr2",
    "gi|371561113|gb|CM000996.2|": "chr3",
    "gi|371561112|gb|CM000997.2|": "chr4",
    "gi|371561111|gb|CM000998.2|": "chr5",
    "gi|371561110|gb|CM000999.2|": "chr6",
    "gi|371561109|gb|CM001000.2|": "chr7",
    "gi|371561108|gb|CM001001.2|": "chr8",
    "gi|371561107|gb|CM001002.2|": "chr9",
    "gi|371561106|gb|CM001003.2|": "chr10",
    "gi|371561105|gb|CM001004.2|": "chr11",
    "gi|371561104|gb|CM001005.2|": "chr12",
    "gi|371561103|gb|CM001006.2|": "chr13",
    "gi|371561102|gb|CM001007.2|": "chr14",
    "gi|371561101|gb|CM001008.2|": "chr15",
    "gi|371561100|gb|CM001009.2|": "chr16",
    "gi|371561099|gb|CM001010.2|": "chr17",
    "gi|371561098|gb|CM001011.2|": "chr18",
    "gi|371561097|gb|CM001012.2|": "chr19"
}

g4['chrom'] = g4[0].map(ncbi_to_ucsc_mapping)


cross_mut_19 = pbw.open("/home/maxime/Nucleoseq/demo/test/crossmut.bw").values("chr19", 0, -1, numpy = True)
mnase_19 = pbw.open("/home/maxime/Nucleoseq/demo/labels/sparse_A_16_gaussiansmoothed.bw").values("chr19", 0, -1, numpy = True)
chem_19 = pbw.open("/home/maxime/Nucleoseq/demo/labels/GSM2183909_Chemical_NCPscore_mm10_SMOOTHED.bw").values("chr19", 0, -1, numpy = True)

In [ ]:

s_p = g4[(g4.chrom == "chr19") & (g4[5] == "+")][1].values
ranges_p = mf.create_ranges(s_p - 1000, s_p + 1000)
ranges_p = ranges_p.reshape((-1, 2000))

s_m = g4[(g4.chrom == "chr19") & (g4[5] == "-")][2].values
ranges_m = mf.create_ranges(s_m - 1000, s_m + 1000)
ranges_m = ranges_m.reshape((-1, 2000))[:, ::-1]

ranges = np.vstack([ranges_p, ranges_m])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def jointplot(sig, ranges, axs, hm_log=False, imshow_kw={}):
    """
    Renders a joint plot into specific subplots.
    axs must be an array of two Axes (one column: [ax_top, ax_bottom]).
    """
    sig = sig[ranges]
    
    axs[0].plot(np.mean(sig, axis=0))
    
    if hm_log: 
        sig = np.log10(sig + mf.eps)
        

    axs[1].imshow(sig, cmap="Reds", aspect="auto", **imshow_kw)

fig, axs_global = plt.subplots(2, 3, figsize=(3, 2), gridspec_kw={"hspace":0, "wspace":0.5})


jointplot(cross_mut_19, ranges, axs=axs_global[:, 0], hm_log=True)
jointplot(mnase_19,     ranges, axs=axs_global[:, 1], imshow_kw={"vmin": 0, "vmax": 1})
jointplot(chem_19,      ranges, axs=axs_global[:, 2], imshow_kw={"vmin": 0, "vmax": 1})

for a in axs_global[1][1:]:
    a.set_yticks([])
    a.set_xticks([])

for a in axs_global[0]:
    a.set_xticks([])

axs_global[1, 0].set_yticks([40_000], [f"{40000:.0e}"])
axs_global[1, 0].set_xticks([0, 1000, 2000], [-1000, 0, 1000])




plt.savefig("/home/maxime/Nucleoseq/notebooks/figures_new/g4.svg")
plt.show()